In [ ]:
# Run this first in a new cell
import torch
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_names = ['fakeV2', 'real']  # adjust if your order differs
num_classes = 2
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

In [ ]:
!pip install -q grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 52.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import files
uploaded = files.upload() #upload downloaded efficientnetb4 model file

Saving efficientnetb4.pth to efficientnetb4.pth


In [ ]:
model_path = '/content/efficientnetb4.pth'

model = torch.load(model_path, map_location=DEVICE, weights_only=False)
model = model.to(DEVICE)
model.eval()

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [ ]:
# ============================================================
# 🌐 AI Image Detector — Single Cell Colab Version
# Run this cell AFTER loading your model from Google Drive
# Requirements: model already loaded, DEVICE, CLASS_NAMES defined
# ============================================================

!pip install -q gradio grad-cam

import os
import torch
import numpy as np
import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import cv2
from PIL import Image as PILImage

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Configuration — change these if needed ────────────────────
CLASS_NAMES = ['fakeV2', 'real']   # must match your training order
THRESHOLD   = 0.27
IMG_SIZE    = 224
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]

# ── Ensure model is float32 ───────────────────────────────────
model = model.float()
for param in model.parameters():
    param.data = param.data.float()
for buf in model.buffers():
    buf.data = buf.data.float()
model.eval()
print(f'✅ Model ready on {DEVICE}')

# ── Grad-CAM ──────────────────────────────────────────────────
target_layers = [model.blocks[-2][-1].conv_pwl]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)
print('✅ Grad-CAM initialized!')

# ── Transform ─────────────────────────────────────────────────
infer_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# ── Prediction function ───────────────────────────────────────
def analyze_image(pil_img):
    if pil_img is None:
        return None, 'Please upload an image.'

    model.eval()
    img_rgb = pil_img.convert('RGB')
    tensor  = infer_transform(img_rgb).unsqueeze(0).to(DEVICE).float()

    # Prediction
    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0].float().cpu().numpy()

    real_idx = CLASS_NAMES.index('real')
    fake_idx = 1 - real_idx
    pred_idx = real_idx if probs[real_idx] >= THRESHOLD else fake_idx
    is_fake  = 'fake' in CLASS_NAMES[pred_idx].lower()

    # Normalized confidence
    raw_prob = float(probs[pred_idx])
    if pred_idx == real_idx:
        normalized = (raw_prob - THRESHOLD) / (1.0 - THRESHOLD)
    else:
        normalized = (raw_prob - (1.0 - THRESHOLD)) / (1.0 - (1.0 - THRESHOLD))
    confidence = float(np.clip(0.5 + normalized * 0.5, 0, 1))

    # Grad-CAM high quality
    targets         = [ClassifierOutputTarget(pred_idx)]
    grayscale       = cam(input_tensor=tensor, targets=targets)[0]
    orig_w, orig_h  = pil_img.size
    grayscale_hires = cv2.resize(grayscale, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)
    grayscale_hires = np.clip(grayscale_hires, 0, 1)
    orig_np         = np.array(img_rgb.resize((orig_w, orig_h))).astype(np.float32) / 255.0
    cam_overlay     = show_cam_on_image(orig_np, grayscale_hires, use_rgb=True)

    # Heatmap figure
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=150)
    fig.patch.set_facecolor('#0d0d0d')
    for ax in axes:
        ax.set_facecolor('#0d0d0d')
        ax.axis('off')
    axes[0].imshow(grayscale_hires, cmap='jet', interpolation='bilinear')
    axes[0].set_title('CAM Heatmap', color='white', fontsize=15, fontweight='bold', pad=14)
    axes[1].imshow(cam_overlay)
    axes[1].set_title('Grad-CAM Overlay', color='white', fontsize=15, fontweight='bold', pad=14)
    plt.subplots_adjust(left=0.01, right=0.99, top=0.93, bottom=0.01, wspace=0.04)

    emoji   = '🤖' if is_fake else '✅'
    verdict = f"{emoji}  {'AI GENERATED' if is_fake else 'REAL IMAGE'}\n{confidence*100:.1f}% confidence"

    return fig, verdict

# ── CSS ───────────────────────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Mono:wght@400;500&display=swap');
:root { --bg:#080810; --surface:#0f0f1e; --border:#1e1e3a; --accent:#e94560; --text:#e8e8f0; --muted:#666688; }
body, .gradio-container { background: var(--bg) !important; font-family: 'DM Mono', monospace !important; color: var(--text) !important; }
.gradio-container { max-width: 1100px !important; margin: 0 auto !important; }
#header { text-align:center; padding:40px 20px 24px; border-bottom:1px solid var(--border); margin-bottom:32px; }
#header h1 { font-family:'Syne',sans-serif !important; font-size:2.8rem !important; font-weight:800 !important; background:linear-gradient(135deg,#e94560,#f5a623); -webkit-background-clip:text; -webkit-text-fill-color:transparent; letter-spacing:-1px; margin:0 !important; }
#header p { color:var(--muted); font-size:0.85rem; margin-top:8px; letter-spacing:2px; text-transform:uppercase; }
.upload-container { border:2px dashed var(--border) !important; border-radius:16px !important; background:var(--surface) !important; transition:border-color 0.3s ease !important; }
.upload-container:hover { border-color:var(--accent) !important; }
#analyze-btn { background:linear-gradient(135deg,#e94560,#c0392b) !important; border:none !important; border-radius:10px !important; color:white !important; font-family:'Syne',sans-serif !important; font-weight:700 !important; font-size:1rem !important; letter-spacing:1px !important; padding:14px 32px !important; cursor:pointer !important; transition:all 0.2s ease !important; box-shadow:0 4px 20px rgba(233,69,96,0.3) !important; width:100% !important; }
#analyze-btn:hover { transform:translateY(-2px) !important; box-shadow:0 8px 28px rgba(233,69,96,0.45) !important; }
#verdict-box textarea, #verdict-box input { font-family:'Syne',sans-serif !important; font-size:1.5rem !important; font-weight:800 !important; text-align:center !important; background:var(--surface) !important; border:2px solid var(--border) !important; border-radius:12px !important; color:var(--text) !important; padding:20px !important; min-height:90px !important; }
.panel { background:var(--surface) !important; border:1px solid var(--border) !important; border-radius:14px !important; padding:16px !important; }
label span { color:var(--muted) !important; font-size:0.75rem !important; letter-spacing:2px !important; text-transform:uppercase !important; }
#info-strip { display:flex; gap:20px; justify-content:center; padding:16px; border-top:1px solid var(--border); margin-top:24px; }
#info-strip span { background:var(--surface); border:1px solid var(--border); border-radius:20px; padding:6px 16px; font-size:0.72rem; color:var(--muted); letter-spacing:1px; }
.plot-container { background:#0d0d0d !important; border-radius:12px !important; }
.plot-container > div { min-height:500px !important; }
.plot-container img { width:100% !important; height:auto !important; }
#footer { text-align:center; padding:20px; color:var(--muted); font-size:0.7rem; letter-spacing:1px; border-top:1px solid var(--border); margin-top:32px; }
"""

# ── Gradio UI ─────────────────────────────────────────────────
with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:

    gr.HTML("""
    <div id="header">
        <h1>⟨ AI IMAGE DETECTOR ⟩</h1>
        <p>EfficientNet-B4 &nbsp;·&nbsp; Grad-CAM XAI &nbsp;·&nbsp; DALL-E Recognition</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            img_input   = gr.Image(type='pil', label='UPLOAD IMAGE',
                                   elem_classes='upload-container', height=280)
            analyze_btn = gr.Button('🔍  ANALYZE IMAGE', elem_id='analyze-btn')
            verdict_out = gr.Textbox(label='DETECTION RESULT', interactive=False,
                                     elem_id='verdict-box',
                                     placeholder='Upload an image and click Analyze...',
                                     lines=2)

        with gr.Column(scale=1):
            cam_plot = gr.Plot(label='GRAD-CAM EXPLAINABILITY', elem_classes='panel')
            gr.HTML("""
            <div style="background:#0f0f1e;border:1px solid #1e1e3a;border-radius:12px;
                        padding:18px 20px;margin-top:8px;font-size:0.78rem;color:#666688;line-height:1.8;">
                <div style="color:#e8e8f0;font-weight:600;margin-bottom:8px;letter-spacing:1px;">
                    🔬 HOW TO READ THE HEATMAP
                </div>
                <span style="color:#e74c3c;">■</span> Red/Yellow regions → areas model focuses on most<br>
                <span style="color:#3498db;">■</span> Blue regions → low attention areas<br><br>
                For AI-generated images, the model typically activates on
                <b style="color:#e8e8f0;">texture artifacts</b>,
                <b style="color:#e8e8f0;">facial distortions</b>, and
                <b style="color:#e8e8f0;">unnatural edges</b> — subtle signs invisible to the human eye.
            </div>
            """)

    gr.HTML("""
    <div id="info-strip">
        <span>MODEL: EfficientNet-B4</span>
        <span>VAL F1: 0.8755</span>
        <span>XAI: Grad-CAM</span>
    </div>
    <div id="footer">
        Final Year Project &nbsp;·&nbsp;
        AI Generated Image Detection using Deep Learning with Explainable AI
    </div>
    """)

    analyze_btn.click(fn=analyze_image, inputs=[img_input], outputs=[cam_plot, verdict_out])

# ── Launch ────────────────────────────────────────────────────
demo.launch(share=True, debug=False)

✅ Model ready on cpu
✅ Grad-CAM initialized!


/tmp/ipykernel_1152/3513364499.py:130: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_1152/3513364499.py:130: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d4ed0b58f3e99033e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
